# Qwen2-Audio

In [1]:
import os
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

/root/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [3]:
CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "Qwen/Qwen2-Audio-7B-Instruct"

os.environ["HF_HOME"] = CACHE_DIR

model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    cache_dir=CACHE_DIR,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Qwen2-Audio model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.68it/s]


Qwen2-Audio model loaded.


In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [5]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="qwen2audio_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [6]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    audio, sr = librosa.load(str(wav_path), sr=processor.feature_extractor.sampling_rate, mono=True)

    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(wav_path)},
                {"type": "text",  "text": USER_PROMPT},
            ],
        },
    ]

    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(
        text=text,
        audios=[audio],
        return_tensors="pt",
        padding=True,
        sampling_rate=processor.feature_extractor.sampling_rate,
    )
    inputs = inputs.to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=64,
    )
    output = processor.batch_decode(
        generated_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output[0]

In [ ]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [8]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True


Pitt-raw:   1%|          | 3/551 [00:02<05:03,  1.80it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control
  DEBUG [1] session=002-1 raw='Control' pred=Control
  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-raw: 100%|██████████| 551/551 [00:38<00:00, 14.35it/s]

[Pitt-raw]
  Accuracy:    0.4392
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 551/551  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True


Lu-raw:   3%|▎         | 2/74 [00:00<00:06, 11.36it/s]

  DEBUG [0] session=F22_000 raw='Control' pred=Control
  DEBUG [1] session=F22_001 raw='Control' pred=Control
  DEBUG [2] session=F26_000 raw='Control' pred=Control


Lu-raw: 100%|██████████| 74/74 [00:06<00:00, 11.54it/s]

[Lu-raw]
  Accuracy:    0.4865
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 74/74  Skipped: 0


In [10]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

[Pitt-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Demucs, exists=True


Pitt-Demucs:   0%|          | 2/551 [00:00<00:36, 15.06it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control
  DEBUG [1] session=002-1 raw='Control' pred=Control
  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-Demucs: 100%|██████████| 551/551 [00:36<00:00, 15.11it/s]


[Pitt-Demucs]
  Accuracy:    0.4392
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 551/551  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Demucs, exists=True


Lu-Demucs:   3%|▎         | 2/74 [00:00<00:04, 16.87it/s]

  DEBUG [0] session=F22_000 raw='Control' pred=Control
  DEBUG [1] session=F22_001 raw='Control' pred=Control
  DEBUG [2] session=F26_000 raw='Control' pred=Control


Lu-Demucs: 100%|██████████| 74/74 [00:04<00:00, 17.06it/s]

[Lu-Demucs]
  Accuracy:    0.4865
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 74/74  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

[Pitt-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Denoiser, exists=True


Pitt-Denoiser:   1%|          | 3/551 [00:00<00:25, 21.38it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control
  DEBUG [1] session=002-1 raw='Control' pred=Control
  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-Denoiser: 100%|██████████| 551/551 [00:25<00:00, 21.73it/s]

[Pitt-Denoiser]
  Accuracy:    0.4392
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 551/551  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True


Lu-Denoiser:   4%|▍         | 3/74 [00:00<00:03, 22.10it/s]

  DEBUG [0] session=F22_000 raw='Control' pred=Control
  DEBUG [1] session=F22_001 raw='Control' pred=Control
  DEBUG [2] session=F26_000 raw='Control' pred=Control


Lu-Denoiser: 100%|██████████| 74/74 [00:03<00:00, 22.21it/s]

[Lu-Denoiser]
  Accuracy:    0.4865
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 74/74  Skipped: 0


In [14]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

[Pitt-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-FRCRN_SE, exists=True


Pitt-FRCRN_SE:   1%|          | 3/551 [00:00<00:25, 21.71it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control
  DEBUG [1] session=002-1 raw='Control' pred=Control
  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-FRCRN_SE: 100%|██████████| 551/551 [00:25<00:00, 21.70it/s]

[Pitt-FRCRN_SE]
  Accuracy:    0.4392
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 551/551  Skipped: 0


In [15]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True


Lu-FRCRN_SE:   4%|▍         | 3/74 [00:00<00:03, 22.09it/s]

  DEBUG [0] session=F22_000 raw='Control' pred=Control
  DEBUG [1] session=F22_001 raw='Control' pred=Control
  DEBUG [2] session=F26_000 raw='Control' pred=Control


Lu-FRCRN_SE: 100%|██████████| 74/74 [00:03<00:00, 22.22it/s]

[Lu-FRCRN_SE]
  Accuracy:    0.4865
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 74/74  Skipped: 0


In [16]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

[Pitt-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer, exists=True


Pitt-MossFormer:   1%|          | 3/551 [00:00<00:25, 21.74it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control
  DEBUG [1] session=002-1 raw='Control' pred=Control
  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-MossFormer: 100%|██████████| 551/551 [00:25<00:00, 21.69it/s]

[Pitt-MossFormer]
  Accuracy:    0.4392
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 551/551  Skipped: 0


In [17]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-MossFormer, exists=True


Lu-MossFormer:   4%|▍         | 3/74 [00:00<00:03, 22.06it/s]

  DEBUG [0] session=F22_000 raw='Control' pred=Control
  DEBUG [1] session=F22_001 raw='Control' pred=Control
  DEBUG [2] session=F26_000 raw='Control' pred=Control


Lu-MossFormer: 100%|██████████| 74/74 [00:03<00:00, 22.23it/s]

[Lu-MossFormer]
  Accuracy:    0.4865
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 74/74  Skipped: 0


In [18]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

[Pitt-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Resemble, exists=True


Pitt-Resemble:   0%|          | 2/551 [00:00<00:37, 14.77it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control
  DEBUG [1] session=002-1 raw='Control' pred=Control
  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-Resemble: 100%|██████████| 551/551 [00:36<00:00, 15.12it/s]

[Pitt-Resemble]
  Accuracy:    0.4392
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 551/551  Skipped: 0


In [19]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Resemble, exists=True


Lu-Resemble:   3%|▎         | 2/74 [00:00<00:04, 16.86it/s]

  DEBUG [0] session=F22_000 raw='Control' pred=Control
  DEBUG [1] session=F22_001 raw='Control' pred=Control
  DEBUG [2] session=F26_000 raw='Control' pred=Control


Lu-Resemble: 100%|██████████| 74/74 [00:04<00:00, 17.05it/s]

[Lu-Resemble]
  Accuracy:    0.4865
  F1:          0.0000
  Control Acc: 1.0000
  Dementia Acc:0.0000
  Valid: 74/74  Skipped: 0
